# HMM Model Results
**Data:** `final_patient_df.csv` — hourly ICU time series for a sepsis cohort (1,503 patients)  
**Model:** Gaussian Hidden Markov Model (`hmmlearn`)  

> **Note:** `model.py` and `evaluation.py` were not uploaded. Their logic has been re-implemented inline below based on the structure in `train_model_1.py`. Cells that substitute for those modules are marked with ⚠️.

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings('ignore')

# ── Config (mirrors train_model_1.py) ──────────────────────────────────────────
DATA_PATH    = 'final_patient_df.csv'
N_STATES     = 4
N_ITER       = 100
RANDOM_SEED  = 10
N_SAMPLE     = 1503   # total patients in cohort
SPLIT_RATIO  = 0.8

# Features fed to the HMM
FEATURE_COLS = ['HR', 'MAP', 'SPO2', 'WBC', 'Lactate',
                'oxygenFlag', 'antibioticsFlag', 'cultureFlag', 'vasoFlag',
                'WBC_missing', 'Lactate_missing']

print('Libraries loaded.')

## 2. Load & Clean Data
*Mirrors `load_and_clean_data()` from `data_processing.py`.*

In [ ]:
def load_and_clean_data(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = 'time'

    # Coerce numeric columns
    for col in FEATURE_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f'Loaded {len(df):,} rows | {df["SubjectId"].nunique():,} patients')
    print(f'Columns: {df.columns.tolist()}')
    return df

df = load_and_clean_data(DATA_PATH)
df.head()

### 2a. Basic EDA

In [ ]:
# Missing data overview
missing = df[FEATURE_COLS].isna().mean().sort_values(ascending=False) * 100
print('Missing % per feature:')
print(missing.round(2))

In [ ]:
# Distribution of ICU stay lengths (hours per patient)
stay_lengths = df.groupby('SubjectId').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(stay_lengths, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Stay length (hours)')
axes[0].set_ylabel('Number of patients')
axes[0].set_title('Distribution of ICU Stay Lengths')

df[FEATURE_COLS[:5]].describe().T[['mean','std','min','max']].plot.barh(ax=axes[1])
axes[1].set_title('Continuous Feature Summary')

plt.tight_layout()
plt.show()
print(stay_lengths.describe())

In [ ]:
# Feature correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Build Sequences
*Mirrors `build_sequences()` from `data_processing.py`.*  
Returns one stacked array for `hmmlearn` plus per-patient lengths.

In [ ]:
def build_sequences(df, feature_cols=FEATURE_COLS):
    """
    Returns
    -------
    sequences      : np.ndarray (N_total_timesteps, n_features)  — no missing-indicator cols
    sequences_full : np.ndarray (N_total_timesteps, n_features)  — all cols including missing indicators
    missing_masks  : list of np.ndarray per patient  — True where value was originally missing
    lengths        : list[int]  — timesteps per patient
    pids           : list       — subject IDs in order
    """
    core_cols    = [c for c in feature_cols if '_missing' not in c]
    missing_cols = [c for c in feature_cols if '_missing' in c]

    seqs, seqs_full, masks, lengths, pids = [], [], [], [], []

    for pid, grp in df.groupby('SubjectId'):
        grp = grp.sort_index()
        core_arr = grp[core_cols].values.astype(np.float32)
        full_arr = grp[feature_cols].values.astype(np.float32)
        mask_arr = grp[missing_cols].values.astype(bool) if missing_cols else np.zeros((len(grp), 0), dtype=bool)

        seqs.append(core_arr)
        seqs_full.append(full_arr)
        masks.append(mask_arr)
        lengths.append(len(grp))
        pids.append(pid)

    sequences      = np.vstack(seqs)
    sequences_full = np.vstack(seqs_full)

    print(f'Sequences shape     : {sequences.shape}')
    print(f'Sequences_full shape: {sequences_full.shape}')
    print(f'Patients            : {len(lengths)}')
    print(f'Avg stay length     : {np.mean(lengths):.1f} hrs')
    return sequences, sequences_full, masks, lengths, pids

sequences, sequences_full, missing_masks, lengths, pids = build_sequences(df)

## 4. Split & Scale Data
*Mirrors `split_and_scale_data()` from `data_processing.py`.*

In [ ]:
def split_and_scale_data(sequences, sequences_full, missing_masks, lengths, pids,
                         n_sample=N_SAMPLE, split_ratio=SPLIT_RATIO, random_seed=RANDOM_SEED):
    # Sample patients (already all 1503, but kept for API compatibility)
    rng = np.random.default_rng(random_seed)
    idx = rng.choice(len(pids), size=min(n_sample, len(pids)), replace=False)
    idx_sorted = np.sort(idx)

    sel_pids    = [pids[i]         for i in idx_sorted]
    sel_lengths = [lengths[i]      for i in idx_sorted]
    sel_masks   = [missing_masks[i] for i in idx_sorted]

    # Reconstruct per-patient arrays from the stacked sequences
    cum = np.concatenate([[0], np.cumsum(lengths)])
    sel_seqs      = [sequences[cum[i]:cum[i+1]]      for i in idx_sorted]
    sel_seqs_full = [sequences_full[cum[i]:cum[i+1]] for i in idx_sorted]

    # Train / test split at patient level
    n_train = int(len(sel_pids) * split_ratio)
    train_idx = list(range(n_train))
    test_idx  = list(range(n_train, len(sel_pids)))

    X_train_list = [sel_seqs[i] for i in train_idx]
    X_test_list  = [sel_seqs[i] for i in test_idx]

    train_lens = [sel_lengths[i] for i in train_idx]
    test_lens  = [sel_lengths[i] for i in test_idx]

    # Scale using training data only
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(np.vstack(X_train_list))
    X_test  = scaler.transform(np.vstack(X_test_list))

    print(f'Train: {len(train_lens)} patients | {X_train.shape[0]:,} timesteps')
    print(f'Test : {len(test_lens)}  patients | {X_test.shape[0]:,} timesteps')

    return {
        'X_train': X_train, 'X_test': X_test,
        'train_lens': train_lens, 'test_lens': test_lens,
        'train_pids': [sel_pids[i] for i in train_idx],
        'test_pids':  [sel_pids[i] for i in test_idx],
        'train_seqs_full': [sel_seqs_full[i] for i in train_idx],
        'test_seqs_full':  [sel_seqs_full[i] for i in test_idx],
        'train_masks': [sel_masks[i] for i in train_idx],
        'test_masks':  [sel_masks[i] for i in test_idx],
        'scaler': scaler,
    }

data_dict = split_and_scale_data(
    sequences, sequences_full, missing_masks, lengths, pids,
    n_sample=N_SAMPLE, split_ratio=SPLIT_RATIO, random_seed=RANDOM_SEED
)

## 5. Build & Train HMM
*⚠️ Substitutes for `build_hmm()` in the missing `model.py`.*

In [ ]:
def build_hmm(n_states=N_STATES, n_iter=N_ITER, random_seed=RANDOM_SEED, scaler=None):
    """
    ⚠️  Inline substitute for model.build_hmm().
    Builds a Gaussian HMM.  `scaler` is accepted for API compatibility
    but hmmlearn operates on the already-scaled data.
    """
    model = GaussianHMM(
        n_components=n_states,
        covariance_type='full',
        n_iter=n_iter,
        random_state=random_seed,
        verbose=False,
    )
    return model

model = build_hmm(
    n_states=N_STATES,
    n_iter=N_ITER,
    random_seed=RANDOM_SEED,
    scaler=data_dict['scaler']
)

print(f'Training Gaussian HMM ({N_STATES} states, {N_ITER} iterations)...')
model.fit(data_dict['X_train'], data_dict['train_lens'])
print('Training complete.')
print(f'Converged: {model.monitor_.converged}')

## 6. Evaluation
*⚠️  Substitutes for `run_evaluation_pipeline()` in the missing `evaluation.py`.*

### 6a. Log-likelihood & Convergence

In [ ]:
train_ll = model.score(data_dict['X_train'], data_dict['train_lens'])
test_ll  = model.score(data_dict['X_test'],  data_dict['test_lens'])

print(f'Train log-likelihood per timestep: {train_ll:.4f}')
print(f'Test  log-likelihood per timestep: {test_ll:.4f}')

# Convergence history
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model.monitor_.history, marker='o', ms=3, color='steelblue')
ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log-likelihood')
ax.set_title('HMM Training Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6b. Transition Matrix & Start Probabilities

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Transition matrix
sns.heatmap(model.transmat_, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=[f'S{i}' for i in range(N_STATES)],
            yticklabels=[f'S{i}' for i in range(N_STATES)],
            ax=axes[0])
axes[0].set_title('Transition Matrix')
axes[0].set_xlabel('To State')
axes[0].set_ylabel('From State')

# Start probabilities
axes[1].bar([f'S{i}' for i in range(N_STATES)], model.startprob_,
            color='steelblue', edgecolor='white')
axes[1].set_title('Initial State Probabilities')
axes[1].set_ylabel('Probability')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### 6c. Emission Means per State
What does each state look like clinically?

In [ ]:
# Inverse-transform means back to original scale for interpretability
core_cols = [c for c in FEATURE_COLS if '_missing' not in c]
means_orig = data_dict['scaler'].inverse_transform(model.means_)

means_df = pd.DataFrame(means_orig, columns=core_cols,
                        index=[f'State {i}' for i in range(N_STATES)])
print('Emission means (original scale):')
display(means_df.round(3))

# Heatmap of z-scored means (relative severity across states)
fig, ax = plt.subplots(figsize=(10, 4))
means_z = pd.DataFrame(model.means_, columns=core_cols,
                       index=[f'State {i}' for i in range(N_STATES)])
sns.heatmap(means_z, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('State Emission Means (z-scored features)')
plt.tight_layout()
plt.show()

### 6d. State Assignment on Test Set

In [ ]:
# Decode hidden states for all test patients
hidden_states = model.predict(data_dict['X_test'], data_dict['test_lens'])

# Overall state distribution
unique, counts = np.unique(hidden_states, return_counts=True)
state_pct = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([f'State {s}' for s in unique], state_pct,
              color=cm.tab10.colors[:N_STATES], edgecolor='white')
ax.set_ylabel('% of timesteps')
ax.set_title('State Distribution — Test Set')
for bar, pct in zip(bars, state_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 6e. Example Patient State Trajectory

In [ ]:
def plot_patient_trajectory(patient_idx=0):
    """Plot vitals and inferred HMM states for one test patient."""
    cum = np.concatenate([[0], np.cumsum(data_dict['test_lens'])])
    start, end = int(cum[patient_idx]), int(cum[patient_idx + 1])

    pid    = data_dict['test_pids'][patient_idx]
    states = hidden_states[start:end]
    X_pat  = data_dict['X_test'][start:end]   # scaled

    X_orig = data_dict['scaler'].inverse_transform(X_pat)
    t      = np.arange(len(states))

    core_cols_plot = [c for c in FEATURE_COLS if '_missing' not in c]
    n_vitals = min(5, len(core_cols_plot))  # HR, MAP, SPO2, WBC, Lactate

    fig, axes = plt.subplots(n_vitals + 1, 1, figsize=(14, 2.5 * (n_vitals + 1)),
                              sharex=True)

    colors = cm.tab10.colors

    for ax_i, col in enumerate(core_cols_plot[:n_vitals]):
        col_idx = core_cols_plot.index(col)
        axes[ax_i].plot(t, X_orig[:, col_idx], color='black', lw=0.8)
        # shade by state
        for s in range(N_STATES):
            mask = states == s
            axes[ax_i].fill_between(t, 0, 1, where=mask,
                                    transform=axes[ax_i].get_xaxis_transform(),
                                    alpha=0.2, color=colors[s], label=f'S{s}' if ax_i == 0 else '')
        axes[ax_i].set_ylabel(col)
        if ax_i == 0:
            axes[ax_i].legend(loc='upper right', fontsize=7, ncol=N_STATES)

    # State sequence as a discrete bar
    axes[-1].scatter(t, states, c=[colors[s] for s in states], s=10, zorder=2)
    axes[-1].set_yticks(range(N_STATES))
    axes[-1].set_yticklabels([f'S{s}' for s in range(N_STATES)])
    axes[-1].set_ylabel('State')
    axes[-1].set_xlabel('Hours since ICU admission')

    fig.suptitle(f'Patient {pid} — Hidden State Trajectory', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_patient_trajectory(patient_index := 0)
# Change `patient_index` to inspect a different test patient

### 6f. Per-Patient Log-likelihood Distribution

In [ ]:
cum = np.concatenate([[0], np.cumsum(data_dict['test_lens'])])
per_patient_ll = [
    model.score(
        data_dict['X_test'][int(cum[i]):int(cum[i+1])],
        [data_dict['test_lens'][i]]
    )
    for i in range(len(data_dict['test_lens']))
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(per_patient_ll, bins=30, color='steelblue', edgecolor='white')
ax.axvline(np.mean(per_patient_ll), color='red', linestyle='--',
           label=f'Mean = {np.mean(per_patient_ll):.2f}')
ax.set_xlabel('Log-likelihood per timestep')
ax.set_ylabel('Number of patients')
ax.set_title('Per-Patient Log-likelihood (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean  : {np.mean(per_patient_ll):.4f}')
print(f'Std   : {np.std(per_patient_ll):.4f}')
print(f'Min   : {np.min(per_patient_ll):.4f}')
print(f'Max   : {np.max(per_patient_ll):.4f}')

### 6g. Intervention Rates by State
Do states separate clinically meaningful treatment patterns?

In [ ]:
flag_cols = ['oxygenFlag', 'antibioticsFlag', 'cultureFlag', 'vasoFlag']

# Reconstruct original-scale test data
X_test_orig = data_dict['scaler'].inverse_transform(data_dict['X_test'])
core_cols_all = [c for c in FEATURE_COLS if '_missing' not in c]
test_df = pd.DataFrame(X_test_orig, columns=core_cols_all)
test_df['state'] = hidden_states

intervention_by_state = test_df.groupby('state')[flag_cols].mean()
print('Mean flag rate by state:')
display(intervention_by_state.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
intervention_by_state.plot(kind='bar', ax=ax, edgecolor='white', width=0.7)
ax.set_xticklabels([f'State {s}' for s in range(N_STATES)], rotation=0)
ax.set_ylabel('Mean flag rate')
ax.set_title('Intervention / Event Rates by State')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Notes & Next Steps

**What's working:**  
- Data loads and sequences build correctly from `final_patient_df.csv`  
- HMM trains without error using `hmmlearn.GaussianHMM`  
- States show interpretable clinical patterns (check emission means heatmap)

**To hook back into the original scripts once uploaded:**  
- Replace cells 5 & 6 with `from model import build_hmm` and `from evaluation import run_evaluation_pipeline`  
- Confirm `data_processing.build_sequences` returns the same tuple signature used here  
- `split_and_scale_data` in `data_processing_fixed.py` likely handles the `n_sample` random draw — verify it uses `RANDOM_SEED` consistently

**Potential issues to check in the original code:**  
- `data_processing_fixed.py` contains raw data loading from hardcoded Windows paths — this needs to be refactored into `load_and_clean_data()` accepting a CSV path  
- The `oxyEventsChart2` / `culturesDf` variable references in the loop (lines 447–451) may shadow outer-scope variables — check for name collisions  